In [ ]:
import os, sys

# GPU 디바이스
os.environ["CUDA_DEVICE_ORDER"] = "PCI_BUS_ID"
os.environ["CUDA_VISIBLE_DEVICES"] = "2"

# 경로
HOME = "/home/j-j14d208"
CODE_DIR = f"{HOME}/2_ml_pipeline"
DATA_DIR = f"{HOME}/kospi200-project"

os.environ["BASE_PATH"] = DATA_DIR
os.chdir(CODE_DIR)
if CODE_DIR not in sys.path:
    sys.path.insert(0, CODE_DIR)

# logger Jupyter 호환 패치
import logging
import utils.logger as _log_mod

def setup_logger(name):
    logger = logging.getLogger(name)
    if logger.handlers:
        return logger
    logger.setLevel(logging.DEBUG)
    fmt = logging.Formatter("%(asctime)s | %(levelname)s | %(name)s | %(message)s", "%H:%M:%S")
    h = logging.StreamHandler(sys.stdout)
    h.setLevel(logging.INFO)
    h.setFormatter(fmt)
    logger.addHandler(h)
    return logger

_log_mod.setup_logger = setup_logger

import torch
print(f"GPU: {torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'N/A'}")
print(f"작업 디렉토리: {os.getcwd()}")

In [ ]:
!pip install pytorch-forecasting lightning --quiet

In [ ]:
"""
N-HiTS 주가 예측 모델 학습
- Neural Hierarchical Interpolation for Time Series
- 순수 MLP 기반, 다중 해상도 분해
- 빠른 학습, 강력한 베이스라인
"""
import warnings
from pathlib import Path

import numpy as np
import pandas as pd

try:
    import lightning.pytorch as pl
    from lightning.pytorch.callbacks import EarlyStopping, LearningRateMonitor
except ImportError:
    import pytorch_lightning as pl
    from pytorch_lightning.callbacks import EarlyStopping, LearningRateMonitor

warnings.filterwarnings("ignore")
pl.seed_everything(42)

print(f"PyTorch: {torch.__version__}")
print(f"Lightning: {pl.__version__}")

In [ ]:
# ========== 설정 ==========
BASE_PATH = Path(os.environ.get("BASE_PATH", "G:/내 드라이브/kospi200-project"))
TFT_FEATURE_PATH = BASE_PATH / "processed" / "tft_features" / "tft_features.parquet"
MODEL_SAVE_PATH = BASE_PATH / "models" / "nhits"
MODEL_SAVE_PATH.mkdir(parents=True, exist_ok=True)

TRAIN_END = "2024-06-30"
VAL_START = "2024-07-01"
VAL_END = "2024-12-31"
TEST_START = "2025-01-01"

# N-HiTS 하이퍼파라미터
MAX_ENCODER_LENGTH = 30
BATCH_SIZE = 128
MAX_EPOCHS = 50
LEARNING_RATE = 1e-3

# N-HiTS specific
N_STACKS = 3               # 스택 수 (다중 해상도)
N_BLOCKS = [1, 1, 1]       # 스택당 블록 수
N_LAYERS_PER_BLOCK = 2     # 블록당 MLP 레이어
HIDDEN_SIZE = 128           # MLP 히든 사이즈
POOLING_SIZES = [2, 4, 8]  # 다운샘플링 비율 (해상도)
DROPOUT = 0.3
GRADIENT_CLIP_VAL = 0.1

print("N-HiTS 설정 완료")
print(f"  스택: {N_STACKS}, 풀링: {POOLING_SIZES}")
print(f"  히든: {HIDDEN_SIZE}, 드롭아웃: {DROPOUT}")

In [ ]:
# ========== 데이터 로드 ==========
print("피처 로드 중...")
df = pd.read_parquet(str(TFT_FEATURE_PATH))
df["date"] = pd.to_datetime(df["date"])
df["sector_id"] = df["sector_id"].astype(str)
df["target_5d"] = df["target_5d"].astype(int)

print(f"  전체: {len(df):,} rows, {df['ticker'].nunique()} tickers")
print(f"  기간: {df['date'].min().date()} ~ {df['date'].max().date()}")
print(f"  타겟 분포: {df['target_5d'].value_counts().to_dict()}")

In [ ]:
train_df = df[df["date"] <= TRAIN_END].copy()
val_df = df[(df["date"] >= VAL_START) & (df["date"] <= VAL_END)].copy()
test_df = df[df["date"] >= TEST_START].copy()

print(f"학습: {len(train_df):,} rows")
print(f"검증: {len(val_df):,} rows")
print(f"테스트: {len(test_df):,} rows")

In [ ]:
# ========== N-HiTS 모델 정의 ==========
import torch.nn as nn
import torch.nn.functional as F

class NHiTSBlock(nn.Module):
    """N-HiTS 단일 블록: 다운샘플링 → MLP → 업샘플링."""
    
    def __init__(self, input_size, hidden_size, output_size, pooling_size, n_layers, dropout):
        super().__init__()
        self.pooling_size = pooling_size
        self.pool = nn.MaxPool1d(pooling_size, stride=pooling_size)
        
        pooled_size = input_size // pooling_size
        layers = [nn.Linear(pooled_size, hidden_size), nn.GELU(), nn.Dropout(dropout)]
        for _ in range(n_layers - 1):
            layers.extend([nn.Linear(hidden_size, hidden_size), nn.GELU(), nn.Dropout(dropout)])
        layers.append(nn.Linear(hidden_size, output_size))
        self.mlp = nn.Sequential(*layers)
    
    def forward(self, x):
        # x: (batch, seq_len)
        pooled = self.pool(x.unsqueeze(1)).squeeze(1)  # (batch, seq_len // pooling)
        return self.mlp(pooled)


class NHiTSClassifier(pl.LightningModule):
    """N-HiTS for binary stock classification."""
    
    def __init__(
        self,
        n_features: int,
        seq_len: int = 30,
        hidden_size: int = 128,
        n_stacks: int = 3,
        n_layers_per_block: int = 2,
        pooling_sizes: list = None,
        dropout: float = 0.3,
        n_classes: int = 2,
        lr: float = 1e-3,
    ):
        super().__init__()
        self.save_hyperparameters()
        self.lr = lr
        self.n_features = n_features
        
        pooling_sizes = pooling_sizes or [2, 4, 8]
        
        # 피처 프로젝션
        self.feature_proj = nn.Linear(n_features, 1)
        
        # N-HiTS 스택
        self.stacks = nn.ModuleList()
        block_output = hidden_size // n_stacks
        for i in range(n_stacks):
            self.stacks.append(
                NHiTSBlock(
                    input_size=seq_len,
                    hidden_size=hidden_size,
                    output_size=block_output,
                    pooling_size=pooling_sizes[i % len(pooling_sizes)],
                    n_layers=n_layers_per_block,
                    dropout=dropout,
                )
            )
        
        # 분류 헤드
        self.head = nn.Sequential(
            nn.Linear(block_output * n_stacks, hidden_size),
            nn.GELU(),
            nn.Dropout(dropout),
            nn.Linear(hidden_size, n_classes),
        )
        
        self.loss_fn = nn.CrossEntropyLoss()
    
    def forward(self, x):
        """x: (batch, seq_len, n_features)"""
        # 피처 합산: (batch, seq_len, n_feat) -> (batch, seq_len)
        x = self.feature_proj(x).squeeze(-1)
        
        # 각 스택의 출력을 모음
        stack_outputs = []
        residual = x
        for stack in self.stacks:
            out = stack(residual)
            stack_outputs.append(out)
        
        # 스택 출력 합산
        combined = torch.cat(stack_outputs, dim=-1)
        logits = self.head(combined)
        return logits
    
    def training_step(self, batch, batch_idx):
        x, y = batch
        logits = self(x)
        loss = self.loss_fn(logits, y)
        self.log("train_loss", loss, prog_bar=True)
        return loss
    
    def validation_step(self, batch, batch_idx):
        x, y = batch
        logits = self(x)
        loss = self.loss_fn(logits, y)
        preds = torch.argmax(logits, dim=1)
        acc = (preds == y).float().mean()
        self.log("val_loss", loss, prog_bar=True)
        self.log("val_acc", acc, prog_bar=True)
        return loss
    
    def configure_optimizers(self):
        optimizer = torch.optim.AdamW(self.parameters(), lr=self.lr, weight_decay=1e-4)
        scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
            optimizer, mode="min", factor=0.5, patience=5, min_lr=1e-6
        )
        return {"optimizer": optimizer, "lr_scheduler": {"scheduler": scheduler, "monitor": "val_loss"}}

print("N-HiTS 모델 클래스 정의 완료")

In [ ]:
# ========== 데이터셋 준비 ==========
from torch.utils.data import Dataset, DataLoader

observed_features = [
    "daily_return", "log_volume_change", "high_low_range",
    "rsi_14", "macd_norm", "bb_percent_b",
    "volume_ratio_5d", "momentum_5d", "momentum_20d",
    "foreign_net_ratio", "inst_net_ratio",
    "kospi200_return", "vix", "vix_change", "usd_krw_change",
    "relative_return",
]
observed_features = [c for c in observed_features if c in df.columns]
N_FEATURES = len(observed_features)

def make_dataset_with_history(full_df, target_start, target_end, seq_len, features):
    """타겟 기간의 샘플을 생성 (encoder 과거 데이터 포함)."""
    samples = []
    target_start_ts = pd.Timestamp(target_start)
    target_end_ts = pd.Timestamp(target_end) if target_end else full_df["date"].max()
    
    for ticker, group in full_df.groupby("ticker"):
        group = group.sort_values("time_idx")
        values = group[features].values.astype(np.float32)
        targets = group["target_5d"].values.astype(np.int64)
        dates = group["date"].values
        
        for i in range(seq_len, len(group)):
            if dates[i] >= target_start_ts and dates[i] <= target_end_ts:
                x = values[i - seq_len:i]
                y = targets[i]
                samples.append((x, y))
    return samples

class StockSequenceDataset(Dataset):
    def __init__(self, samples):
        self.samples = samples
    def __len__(self):
        return len(self.samples)
    def __getitem__(self, idx):
        x, y = self.samples[idx]
        return torch.tensor(x), torch.tensor(y)

train_samples = make_dataset_with_history(df, "2008-01-01", TRAIN_END, MAX_ENCODER_LENGTH, observed_features)
val_samples = make_dataset_with_history(df, VAL_START, VAL_END, MAX_ENCODER_LENGTH, observed_features)
test_samples = make_dataset_with_history(df, TEST_START, None, MAX_ENCODER_LENGTH, observed_features)

train_dataset = StockSequenceDataset(train_samples)
val_dataset = StockSequenceDataset(val_samples)
test_dataset_pt = StockSequenceDataset(test_samples)

train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True, num_workers=0)
val_loader = DataLoader(val_dataset, batch_size=BATCH_SIZE * 2, shuffle=False, num_workers=0)
test_loader = DataLoader(test_dataset_pt, batch_size=BATCH_SIZE * 2, shuffle=False, num_workers=0)

print(f"학습: {len(train_dataset):,} samples")
print(f"검증: {len(val_dataset):,} samples")
print(f"테스트: {len(test_dataset_pt):,} samples")
print(f"피처: {N_FEATURES}개")

In [ ]:
# ========== 모델 초기화 ==========
model = NHiTSClassifier(
    n_features=N_FEATURES,
    seq_len=MAX_ENCODER_LENGTH,
    hidden_size=HIDDEN_SIZE,
    n_stacks=N_STACKS,
    n_layers_per_block=N_LAYERS_PER_BLOCK,
    pooling_sizes=POOLING_SIZES,
    dropout=DROPOUT,
    n_classes=2,
    lr=LEARNING_RATE,
)

total_params = sum(p.numel() for p in model.parameters())
print(f"모델 파라미터: {total_params / 1e3:.1f}K")

In [ ]:
# ========== 학습 ==========
early_stop = EarlyStopping(monitor="val_loss", patience=8, verbose=True, mode="min")
lr_monitor = LearningRateMonitor(logging_interval="epoch")

trainer = pl.Trainer(
    max_epochs=MAX_EPOCHS,
    accelerator="gpu" if torch.cuda.is_available() else "cpu",
    devices=1,
    gradient_clip_val=GRADIENT_CLIP_VAL,
    callbacks=[early_stop, lr_monitor],
    enable_progress_bar=True,
    log_every_n_steps=50,
)

print("N-HiTS 학습 시작...")
trainer.fit(model, train_dataloaders=train_loader, val_dataloaders=val_loader)
print(f"학습 완료! Best val_loss: {early_stop.best_score:.4f}")

In [ ]:
# ========== 성능 평가 ==========
from sklearn.metrics import accuracy_score, roc_auc_score, classification_report

best_model = NHiTSClassifier.load_from_checkpoint(trainer.checkpoint_callback.best_model_path)
best_model.eval()
best_model.cuda()

all_probs, all_labels = [], []
with torch.no_grad():
    for x, y in val_loader:
        x = x.cuda()
        logits = best_model(x)
        probs = torch.softmax(logits, dim=-1)[:, 1].cpu().numpy()
        all_probs.extend(probs)
        all_labels.extend(y.numpy())

val_probs = np.array(all_probs)
val_actuals = np.array(all_labels)
val_preds = (val_probs >= 0.5).astype(int)

da = accuracy_score(val_actuals, val_preds)
try:
    auc = roc_auc_score(val_actuals, val_probs)
except:
    auc = float("nan")

print("=" * 60)
print("  N-HiTS 검증 성능")
print("=" * 60)
print(f"  Direction Accuracy (DA): {da*100:.2f}%")
print(f"  AUC-ROC: {auc:.4f}")
print(f"  양성 예측 비율: {val_preds.mean()*100:.1f}%")
print()
print(classification_report(val_actuals, val_preds, target_names=["하락", "상승"]))
print(f"\n비교: LSTM DA=48.9%, LightGBM DA=50.8%")

In [ ]:
# ========== 테스트 예측 ==========
test_probs_list, test_labels_list = [], []
with torch.no_grad():
    for x, y in test_loader:
        x = x.cuda()
        logits = best_model(x)
        probs = torch.softmax(logits, dim=-1)[:, 1].cpu().numpy()
        test_probs_list.extend(probs)
        test_labels_list.extend(y.numpy())

test_probs = np.array(test_probs_list)
test_actuals = np.array(test_labels_list)
test_preds = (test_probs >= 0.5).astype(int)

test_da = accuracy_score(test_actuals, test_preds)
try:
    test_auc = roc_auc_score(test_actuals, test_probs)
except:
    test_auc = float("nan")

print("=" * 60)
print("  N-HiTS 테스트 성능 (OOS)")
print("=" * 60)
print(f"  Direction Accuracy (DA): {test_da*100:.2f}%")
print(f"  AUC-ROC: {test_auc:.4f}")
print()
print(classification_report(test_actuals, test_preds, target_names=["하락", "상승"]))

In [ ]:
# ========== 저장 ==========
import json
from datetime import datetime

trainer.save_checkpoint(str(MODEL_SAVE_PATH / "nhits_best.ckpt"))

pred_df = pd.DataFrame({"prob": test_probs, "actual": test_actuals, "pred": test_preds})
pred_df.to_parquet(str(MODEL_SAVE_PATH / f"nhits_predictions_{datetime.now().strftime('%Y%m%d')}.parquet"))

metrics = {
    "model": "N-HiTS",
    "val_da": round(da * 100, 2), "val_auc": round(float(auc), 4),
    "test_da": round(test_da * 100, 2), "test_auc": round(float(test_auc), 4),
    "config": {"hidden_size": HIDDEN_SIZE, "n_stacks": N_STACKS, "pooling_sizes": POOLING_SIZES},
    "timestamp": datetime.now().isoformat(),
}
with open(str(MODEL_SAVE_PATH / "nhits_metrics.json"), "w") as f:
    json.dump(metrics, f, indent=2)

print(f"저장 완료: {MODEL_SAVE_PATH}")

## N-HiTS 결과 요약

| 모델 | DA | AUC | 비고 |
|------|-----|-----|------|
| LSTM | 48.9% | - | 시퀀스 20일, 5피처 |
| LightGBM | 50.8% | - | 60+ 피처 |
| TFT | ?% | ? | Transformer + static |
| PatchTST | ?% | ? | 패치 기반 Transformer |
| **N-HiTS** | **?%** | **?** | MLP 다중 해상도 |

### 특징
- 순수 MLP 기반 (Transformer/RNN 없음)
- 다중 해상도 분해: 단기/중기/장기 패턴 분리
- 매우 빠른 학습, 적은 파라미터